# v2 REAL — 개인 의도 보정과 XYZ 배율

기존 strict JSON worker를 사용합니다. 처음에는 DRY RUN이고, PC에서 REAL RUN을 선택하면 실제 명령이 전송됩니다. 먼저 SIM에서 개인 보정의 수평/상하/대각선 동작을 확인하세요.

위치 매핑: `operator delta → C → XYZ gains → 20cm 작업범위 → IK`. Z를 고정하거나 전진 궤적으로 강제 투영하지 않습니다.
기준 자세는 로봇의 현재 EEF와 편한 컨트롤러 자세를 양손 grip 시점에 대응시킵니다. 좌표는 로봇 전방 +X, 왼쪽 +Y, 위 +Z입니다.


수동 관절 조정: 조작을 멈춘 뒤 관절과 절대 목표 각도(°)를 선택하고 **관절 미리보기 → 조정 실행**을 누릅니다. 같은 창의 **Cancel return**으로 중단하고, 조정 후 **Initial pose**로 복귀할 수 있습니다. 최초 HOME 목표는 보존됩니다.
그리퍼는 기본 60°/s, 새 bridge 사용 시 10~120°/s를 입력할 수 있습니다. REAL 적용 전 로봇 PC의 `inbound.py`와 `fixed_quest_protocol.py`를 동일 버전으로 교체하고 inbound/outbound를 재시작하세요. HOME과 수동 조정은 최대 10°/s입니다.


In [ ]:
# Locate this standalone workspace from a notebook or repository directory.
from pathlib import Path
import runpy

_v2_candidates = [candidate for base in (Path.cwd(), *Path.cwd().parents)
                  for candidate in (base, base / "VR_teleoperation")]
_v2_root = next((candidate for candidate in _v2_candidates
                 if (candidate / ".project_v2").is_file()), None)
if _v2_root is None:
    raise RuntimeError("Open this notebook from the VR_teleoperation workspace.")
_v2_bootstrap = runpy.run_path(str(_v2_root / "package" / "init_project.py"))
for _v2_name in _v2_bootstrap["__all__"]:
    globals()[_v2_name] = _v2_bootstrap[_v2_name]


import numpy as np


## 설정 — 사용자 SIM 확인을 반영한 v1 REAL 시험

사용자가 SIM 조작을 확인하고 REAL 시험을 요청한 v1 후보를 지정했습니다. 원본 accepted=false를 변경하지 않으며, 별도 SIM 확인 기록을 프로필·시연의 해시에 연결합니다. 다른 후보로 바뀌면 기존 확인 기록으로 로딩되지 않습니다.

XYZ 모두 0.5, 팔 속도 상한 60°/s, Initial pose 10°/s입니다. 기존 관절·충돌·속도·피드백 제한을 유지합니다. 기본 DRY RUN에서 연결과 피드백을 확인한 뒤 REAL RUN 버튼으로 실제 명령 전송을 시작합니다. 일반 통과 프로필을 사용할 때는 REAL_CANDIDATE_REVIEW_PATH=None으로 설정합니다.


In [ ]:
from package.intent_teleop import IntentTeleopSettings, selected_profile, build_models, run_intent_teleop, tracking_mode_text
settings = IntentTeleopSettings(
    backend="real",
    PROFILE_PATH=PROJECT_DIR / "calibration/diagnostics/v1.json",
    REAL_CANDIDATE_CAPTURE_PATH=PROJECT_DIR / "calibration/diagnostics/v1_trajectory.json",
    REAL_CANDIDATE_REVIEW_PATH=PROJECT_DIR / "calibration/reviews/v1-sim-reviewed-real-trial.json",
    CONFIRM_CALIBRATION_SETUP=True,  # 본인·Quest 방향·작업 위치를 확인한 후 True
    POSITION_SCALE=.5, POSITION_SCALE_Y=.5, POSITION_SCALE_Z=.5,
    MAX_JOINT_SPEED=np.deg2rad(60.),  # 실행 창에서 10~120 숫자 입력 + 적용
    MAX_GRIPPER_SPEED=np.deg2rad(60.),  # 그리퍼 전용 상한. 새 worker에서 10~120 숫자 입력 + 적용 (복귀 중 10°/s)
    IK_RECOVERY_TIMEOUT_S=2,  # IK 미수렴 시 최대 2초 유지·재시도 (0: 즉시 중단)
    NULL_REFERENCE="self_motion",  # J3·5 비틀림 억제: self-motion 좌표만 초기 자세로 고정 ("previous": 기존 동작)
    QUEST_HOST=None,  # 자동 LAN IP가 틀리면 PC IP를 문자열로 입력
    TRACKING_PAUSE_S=.1, ACCEPT_EMULATED_POSE=True,  # 시작 모드: 추정 위치 사용 (컨트롤러가 사라지면 0.1초 뒤 정지)
    TRACKING_MODE_BUTTONS=("emulated", "pause_0.5"),  # 실행 창 버튼: 추정 위치 사용 / 정지 0.5초
    DRY_RUN=True,
)
settings.validate()
profile, matrices = selected_profile(settings)
print("개인 보정:", "미적용 — identity" if profile is None else profile["profile_id"])
print("XYZ:", settings.POSITION_SCALES["left"], "| 속도:", np.rad2deg(settings.MAX_JOINT_SPEED),
      "| 그리퍼:", np.rad2deg(settings.MAX_GRIPPER_SPEED), "| null 기준:", settings.NULL_REFERENCE)
print("사용자 SIM 확인 후 REAL 후보 시험 | 자동 검증 accepted:", profile["accepted"])
print("추적 손실 시작 모드:", tracking_mode_text({"tracking_pause_s": settings.TRACKING_PAUSE_S,
                                                 "accept_emulated": settings.ACCEPT_EMULATED_POSE}))


## 모델 준비 확인 (통신 없음)

기존 FK·IK와 관절/충돌 검사를 그대로 사용합니다. 아래 셀은 모델만 확인합니다.


In [ ]:
models = build_models(settings)
from utils import get_sg2_wrist_control_T
for hand in ("left", "right"):
    print(hand, "EEF (m):", get_sg2_wrist_control_T(models["env"], hand)[:3, 3])
del models


## 실행

[Worker 실행 안내](../docs/WORKER_SETUP.md)에 따라 bringup/outbound/inbound를 실행하세요.

- 다른 Quest 수신 노트북을 STOP하여 8443 포트를 비우세요.
- 양손 grip으로 시작하고, 한쪽이라도 놓으면 양팔이 멈춥니다.
- XYZ는 grip을 모두 놓고 숫자 입력 후 적용합니다. 새 grip 기준에서 반영되므로 목표가 갑자기 바뀌지 않습니다.
- 속도는 실행 중 입력 후 적용할 수 있습니다. 감속 시 이전 명령 ACK를 기다릴 수 있습니다.
- 그리퍼 속도는 팔 속도와 별개로 실행 중 10~60°/s로 적용할 수 있습니다. 로봇 bridge가 그리퍼를 60°/s로 제한하므로 그 이상은 입력할 수 없고, Initial pose 복귀 중에는 복귀 속도를 따릅니다.
- Initial pose는 Quest/그립 없이 이번 실행에서 처음 측정한 관절 자세로 10°/s 복귀합니다. worker를 재시작해도 이 목표는 바뀌지 않습니다.
- 누르는 순간 복귀 전체 경로를 기존 관절·5mm 충돌 검사로 확인합니다. 직선 경로가 막히면 J3·J5 먼저 풀기 → 전완·손목 먼저 풀기 → 오른팔 먼저 → 왼팔 먼저 순서로 단계 경로를 찾고, 명령은 확인한 경로 위로만 보냅니다. 모든 경로가 막히면 아무 명령도 보내지 않으므로, VR 조작으로 팔 사이를 벌린 뒤 다시 누르세요. 중간에 멈춘 복귀도 다시 누르면 현재 자세에서 새로 검사합니다.
- **추적 손실 모드**: 기본은 `추정 위치 사용`입니다. 한 손의 광학 추적이 끊겨도 Quest 추정 위치로 조작을 이어가고, 컨트롤러가 아예 사라지면 0.1초 뒤 정지합니다. 추정 위치는 손실 순간 수 cm~십수 cm 튈 수 있습니다.
- 실행 창에서 `정지 0.5초`로 바꾸면 추적 손실 중 현재 명령을 최대 0.5초 유지하고, 그 안에 추적이 돌아오면 같은 grip 기준에서 이어갑니다. 모드 전환은 양손 grip을 놓은 상태에서 적용됩니다.
- 두 모드 모두 한 프레임 20cm 초과 점프·grip 해제·피드백/실행 오류는 정지하고, grip당 20cm 작업범위와 속도·충돌 제한은 그대로입니다.
- 일시적인 IK 미수렴은 양팔·그리퍼의 현재 명령을 최대 1초 유지하며 최신 목표로 재시도합니다. 복구되면 같은 grip 기준에서 조작을 이어갑니다. 화면에 경과 시간과 재시도 횟수가 표시되며, 시간 초과·그립 해제·피드백 이상은 정지합니다.
- 팔꿈치 양옆의 두 회전(Joint 3·5)이 조작과 무관하게 감기지 않도록, EEF가 볼 수 없는 자세 성분만 시작 자세 쪽으로 유지합니다. 목표 추종에 쓰이는 성분은 기존과 같이 IK가 결정합니다.
- 로그는 `.cache/fixed_quest/recordings/*-intent-real.jsonl`에 기록됩니다.


In [ ]:
log_path = run_intent_teleop(settings)
